# MandiaCompact S2S — étape 5 : phrases quotidiennes
Continuation du meilleur modèle hybride de l’étape 4 (pas 1250), tokenizer et architecture de 13,8 M paramètres conservés. AdamW redémarre volontairement pour cette nouvelle phase ; les reprises de l’étape 5 conservent ensuite son état complet.

393 paires nouvelles issues des exemples du Wiktionnaire via Kaikki, CC BY-SA 4.0, avec URLs, identifiants de sens et exemples bruts. 277 train, 64 validation, 52 test, regroupement par lemme et alias exacts. Relecture technique par Mandia, aucune validation native revendiquée. Aucune phrase baoulé générée pour servir de cible.

600 étapes, graine 42, LR 2e-5. Mélange 60% nouvelles phrases, 30% rappel de quatre domaines précédents, 10% anciennes réponses simples. Sélection sur validation nouvelle/ancienne et conservation des 14 réponses simples. Test avant/après consulté pour comparaison, jamais pour sélectionner.

Le mode guidé « Dis en baoulé : ... » demande une traduction ; ce n’est pas un corpus de dialogues libres. Les nouveaux textes bibliques complets restent indexés : aucune licence d’entraînement n’est supposée à partir de leur accès en lecture. L’expérience garde les extraits bibliques admis de l’étape 4.

Tous les poids, AdamW, RNG, tokenizer source et rapports restent dans Google Drive. Chaque 100 étapes crée un checkpoint complet.


# MandiaCompact S2S — étape 5 exécutée

393 paires français–baoulé préparées (277 train,64 validation,52 test), provenant des exemples du Wiktionnaire via Kaikki. Cibles baoulé recopiées exactement, CC BY-SA 4.0 et URLs conservées. Architecture/tokenizer conservés, 13 782 528 paramètres.
600 mises à jour supplémentaires, meilleur checkpoint de validation : étape 400. Départ du meilleur modèle de l’étape 4, avec un nouvel AdamW pour cette phase ; les reprises suivantes restaurent l’état complet. 467986 tokens réels et 204153 tokens supervisés.

| Mesure | Avant | Après |
|---|---:|---:|
| Perte sur 52 nouvelles références ↓ | 5.720 | 4.146 |
| chrF++ traduction ↑ | 9.86 | 14.93 |
| chrF++ réponse guidée ↑ | 9.62 | 15.16 |
| Correspondances exactes sur les nouvelles traductions | 0/52 | 0/52 |
| Anciennes réponses simples conservées | 14/14 | 14/14 |
| Perte sur le test précédent ↓ | 3.650 | 3.621 |

Les deux modes utilisent les mêmes 52 références : ils ne représentent pas 104 phrases indépendantes. Réponse guidée signifie une demande « Dis en baoulé : ... », pas une conversation libre. Test consulté avant/après pour comparaison, sans choix sur le test. Une seule graine, aucune validation native Mandia.
La relecture des 52 sorties relève 13 cas où le contrôle textuel ne retrouve pas un nom de la référence (orthographes Kouadjo/Kouadio considérées comme équivalentes), et une sortie contenant MediaWiki hors sujet. Le score s’améliore, mais le sens reste souvent incorrect. Le candidat reste expérimental ; aucune maîtrise générale du baoulé n’est annoncée.

Bible : éditions 1974 et 1998, textes et audio repérés. L’importeur par verset est préparé et testé (doublons, éditions, regroupement par chapitre, preuves de réutilisation). Aucun corpus biblique complet nouveau n’a été ajouté ; les extraits Gospel admis de l’étape 4 restent dans le rappel.
Manuel de conversation : 282 pages avec couverture ERIC ; vrais dialogues français/baoulé/anglais, mais OCR des tons/voyelles nasales et licence d’origine encore à vérifier. Zenodo : 565 clips/560 transcriptions, source Glosbe, pas de labels français ; non admis à cette expérience.
Les modèles exportés et checkpoints complets restent dans Drive. Aucun poids ou fichier audio lourd n’a été téléchargé sur la machine locale.


In [ ]:
%pip -q install "tokenizers==0.22.1" "sacrebleu==2.5.1"


In [ ]:
import pathlib,sys,json,gzip,base64,hashlib
from google.colab import drive
drive.mount('/content/drive')
PARENT=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape4')
OUT=PARENT.parent/'mandia-compact-s2s-etape5'
assert (PARENT/'hybrid/model.pt').exists(), 'Le meilleur modèle de l’étape 4 doit être sauvegardé dans Drive.'
OUT.mkdir(exist_ok=True)
ASSETS={'quotidien-prepare.jsonl': '...', 'AUDIT-DONNEES.json': 'H4sIAAAAAAAC/31UTY/bNhC951cMDOTmD1m2tXJ8yrpJ0UtbpEWBnAiKHMmEKVIlJXs3wf73Dklp15sucpTe8M2bxzf8/g5gJk4oziiZ5D3OPsAsz/Jisc4W681sHnDHr0wq0StruHtkaHqn0FNlmRex4Ab0J57visBSC5T7TIqskJsdLypRrTOZiWJXllW53fBtmZW4vauErLO84uVdUW3LjSxkXW72gSNSCzuYPvT6Tl+jFnzgbaejgm22nSdAeatJv2S1401LGgnNR2wwQiN3rOXuPHTMOnbhTvFEPBG0yntlGtY7bjxR0UCEbkaUeoqeyaHTSiSbcgKeosbOYccdtXb2Ghg3+3hq5qn4Vjsxq8CZ392NrBeulZxaFZOSHn1Qv3vpYPDKeu4a7FmDBt10ZGaswQPccztohFThQdhOoYQoWT9C7WwL3g5O4BI+Qq0eCPzs0IgTNIOSKBcO/x2oKdAkBIPywDsST3W9hSh7Di9agRsCqH4Jv4/wKnxOpGQG706Oe4QXtct0n42zQ0c2B/HXkyXVGtuW05kGodODT7InqtBpnI7aE6M/gNUS6KS0Qx+qhR5ohAOQRzcak6DE1eNDD8b2ROGQy0dQJpIEnVqjBjrCk7xkExucDgJPfd/5D6vVmavzWS2ta1a1u6rzGPZVVPb+uHn/cb9K5w01vyB7da811x4j+g2Nlfb/gUifId0orJNkTwzmbsxvTLCiG0rZFE51KbnbfcSfXufmbbb1evNzNip4zfaMpjHept0Vu5/T7oos0T6HObnA4vCMy1b1tLZv+MToyuJ7dDwu7r8utssMWux5uKw5vIiACzpVU1bn8Eqyj1c+3kQIiHWqUYZu/BGoiHeU7rgav2rrK6RQhCQRI9yrivI2vSOUOGOnPGpeofYhQN1QaeVPRPL3X/8c4Iwd5Yv2xhAoMayY7xDDEYrcPGpR5kJ8eBO855csjl2FvqwetGaq7az70ZaWm4Hrty0bscmyvMzhz18+x7XyN6N9+vLbkd4HsmwOldL0i06Nw60+mSaMBA0ZQpsWTCbwuOhOYZWl4to29Ews4Y/jF9qZNnIb7snib+PS0XMU8NFpoMeSmDFOT+Hko2UH8KhRJCsCyUX5IV6MMuRaBMLmj04m97iU6T2Kzov07oSZQiCWs3dP/wFYSAW2zAYAAA==', 'CREDITS.txt': 'H4sIAAAAAAAC/02R3U7CQBCF732K8wBtgcZ4gVdYNDH+xIDGeDlsB5jQ7ja7U5C3d0oQvdpk9pzJOd+88gHK34qalKZwwWuUVa8hJmiAbhkPkb3b4lN2KsFTPGbYC+GJZLcTjHCwD9sQyektqgp3X/lyhutiXOCezNiRRERWEp8gmtBw2xI+Fs8ZEvvEeJxniHQAf1PbNQzytRn2YtmSkvapwBtteLAkbGRvCuc4nRL+TzzaSrL3WGAZ+ugYyVOXtkGnWMfDJT/qvu1QjsubfDLOx5Pst8u5han+fssClSjj3XLgqwmefXZicRZP/4EBJbxYZfGcL5hqWlmZpcbeaR+5xtwYZ3he3FfD/rK4qkLNU4QoG/HUmNfXRtbZNMOss02clwPHN7Ib6ABjw9enU2HH3A3nMbYXfyPOTsWG6zWgtS0N1k1vo6Np4EkHdHuOshZHp5quIWm5Lq5+AIj+/rsIAgAA', 'wiktionnaire-source.jsonl': '...', 'mandia_stage5.py': 'H4sIAAAAAAAC/7Va/27bSJL+X0/BDTAgabdpS7GNhBouzjfxDAJkMkGSyx0gCARFtiSuKZLDJmVrDb/PznPkxe6r6uYPyUpudnH7hy2yu6u7uqq66qtqvnjx4qcir9O8ieq0yK04XWRf/5BW0lgbmWaZbCprUyRf/5FJ69Pkk7DioiobZeVFs5VRY62qoim//uGNri8urDJSwqqLO5mnf5fVeVTF67SWcd1U0krzeB3lq69/KGtqxWsZ35VFmtcKE27KTOIhiXJlvanSrfRG76Syqq9/lEWu8LRq0gRMYYjcRHki89pqcmnVVZQ0MbEtLAlWwZW0FHZDjGAs/tIoK1YNnrJ0UUnljV68eDFKN2VR0aB6jWbxN5CIdaT4pcL8xUbU6UaKDQaIVSxquSmXaSZFXJS70bIqNuC1juIsUsSdmS5SSRrX7eR1gc3rsZ1AuqGf25Z2NO0qjUJVRyt5iZksdalp2w75+wR/Lf2v3ErqwB/Ut0xXo4+//fY5UJce/XplVEFI57YmPyMRR3F9pibqTNZRKa/s0afPtx8+iXcfAyhOTOTZ1SiRS0vt8thJ5DaNpeuPrHRp6Rev3pUyCOy4SSLb58159OwRwboqaDeOy1PIPC4S6VTFvRLYOk3DRqKCxzv/0YbScpVFtbT92VzYpFqZ0POTtSwq6w6GYs1oVJrbwt5GWZqwZeKllqq2509T+RBnDaiC2XzERBUR0YJYy6qjaiXrAEt7LSszG2vey8qeu16aqNPZBIRMWUfqTsC+pSgh7rLm1Z0Bk+KlALnuBLlwWo7FpbDfpArbtRZR0eDQWL5lnw4Hz4kdGGRTxTKY0Srz0wFbetyQI5J3JnNHk7h/HV9fWGCSmvSu0PR64rf796KylHniPNopJIiV8QOR0p6gI/x/cqexPtySZoeEArJRpzK7DfSPgMVD3IEtt7LaJdHOFkQd0D+XCLX+ZlhBlVmKrc1n1DfvGdAc277+JR6IXeKCfoWNpcFhcf/kao3xPCRsPbU/UEffOuNRc5+OGax+r1EzIBx+E4bTSsLVtNSiFdPoP7S9pvlS4ljEMiRVGGvN5X2YFUo51JYJQ9qfAG72JOzQcadFUwePT6PecuDdNqpn2ON3x2UzLOooE3z2VXDhiQuzxZRtFZ5QOheCNMs0rhhfM5kViYXYiZCO8iKq47Xun6V+ejq+nrec0cj7tF7DUXikYvlQY9mtyIlusB+ejUczP6fBMiui2tm6U83ZaZCjE/vS8gwebSKmE47R53oMaw9tA2Zt3UPj6PepE31Lr5qN8zDTL3Pe9wPtGwt5ECUcsuO65zQhWlxtsqDCy9N3tQX/X5idFVlC0h2oyrAAASgEKRmuZC4r9h09SetZ5jO7kmW2w3npJxGTS3fE6/AgQ6VKGafLNOaZYFXEPzyDSNIKoa2odkKlqzyiINfOo2oJd6c9bFYF7z4KPlnB1YVQ0VaG+m0Mx4sIKKuqKeswqoP3Rc4b6Wb2Nnd4drQ3V8HnqpGw6lTVYXHHb9AsglAddBTndh9az6jLgyeaLuQ3x1AXjRlZRUlOk0SPp3Tj3STR5r+1DCieRBsJVqE32lFWiQUCiQoc77XwXr9yxb1MV+s6TGQc7QLvYgzOVma6X7QeispxPQSkJsoQz2TiXE7cKUkqqGSUBaopZbVNFdz6xVRmUclP3nSN7YJp+HreBqvWWLENG7Hd6SJSMktzyfLjmMUbZ0GZwxgbVkCWONSL2F7i3GulIqyVjW22oMIiz3ZaulPjeWI4vlbH9jwIuhdMrQVEE1P4rmXIHhYU3IEoMIU4j/WzlAkE0JiVp2StB1CfiggTVWGVr6hXM08j0NCP4uZ2jAYuNKjrL3f1usjNgJH1Z6L5cIkwyjKahnr6SUhjgjQmeo0Joy9hlCU6TYlOOZjpbj4M8TQRgjpNFRp3Iux+zr7tHnzAYuDmEno1a+CJVgl5GXoxC4UDyACxQAdWDLbLCtbu2B9vP3x8++k2vP188+H2yuaTKpZZo9bteRoiIJhnVOFUAAx64GsZxkVDBxbeyGK0hMPsJObU7lPGcKW9zfi9i7Bjxmu2r/GiOWDxcgU3yCLxacI2gobsSkBPP8LYlK9pBuYE2t6cfDK4/c6hPfkrb9XZGrvy1oqMJaz27AwjOgMYmsrqmam43zAwYHMlLQJ7A4P0jb2uWnt1903BZxM7Yg/+wOz2TcM3Vnh6RF9nrMneePzWUodG5Pdme9Sc/LbxibR96cGnbdI4ZKForxLDZLp2yiucgdcF2FohB1EeddgCaPgbR2JP9f/CCRlsaU6AC2phPxtckJl2J7KNZCbOcfQZ+Nh2HNiSmeQ0q520dcoD/MeWe7EH2HtxDaWipUXLiMfv2XM346FuaDY6dxcGSg4AFZ9mFhsfSD2tjuXaDeN4FOEKaaNDnq4uwhybbk++xWAhcAC03POXFyS29Ef8svl6k1Pv1Yl3deKMTykvBOoCxqKHMj1x0rOXF+75Jnpwxnp9enddA/nKFWMfrM9xNDSYt1wBIMH7I5qe8NKExBZ/I1FvJUJfCxgJconchCeaaR+ROOkQOOY6mLbgkeDhuiiUdNpp4Ge+jyEJnoncZFFH0KTwLq5ck6zkhcl0vVQt0xxrOEzv4njDRK2fKU6n+eoDYY3bqgIGsGFnZ3qsxfCQ53L07k6Y+tys7mIT8d19VCWkvlYyp4EZa6AsUXgJ0Ai2C1BpaKckB6Bb+pn2ZwYtup8ROjGe515Tp5ny4iwt2TZgFNUmPIZ8xp6QtIkwXZLl6F0YqKCdriwdE2nZjH7QSC/gNA4NgA5kHawvuT08ge15cb91vjCD0AAWMg07icDjt4/7B1BucVhM5NMRL/xy8+7tm5vPb397bwuajnyRlzSbUjly6+6HQt6H3M5scLVKFxlOvgWvzU3PfMKP3QHl3Vm9Izk6/F/2CCyD5z5hKPQBxj6UfPc2BN4++xMi1bM8H2BSCw0waXQyBJTM+p8DlMfRIs84QIx61HedVZdv7QME7fpZUGZSfpl/M849D4j/9ij8pHMs6Cdd7kJExGYjGaaXFRxT0SjyOm9uv7z96XbaPRl5a9flsHwhBcCmoCuBOVuIfxEqgKDg5QT52Sa4fCXWMkpUcC3utqF+nAhdf6nCLNrhXAdjOMS9holYLvOQ6K8vRVIVJeX9HlkHO9Fg1tc7Zi/FK/FaTOZ9yWM2vhDjMZp0QSvsw9OVOze4EQ6c/bbR4sx45wk8TDtGbp2NWHRDHp8dIX/iif5k+mQXwCV1taPzx369rVt6nyXVDaNq96ZFIwBrkbLqhM9qnQSmFup9wK9TJ6ZiQBIfZmzj14M8w7xvgq4a6RCKnZrcmfNmSpdFnZzbyybL2gqe6JRqEuVXJj2+HGbHl//vPGhL+ye52M/QNVP/xuV4/mjoXFrpnR9J7I95mMUhsebjT5LT8h0gjTqX1Jbg9NTyd2jD6XuBYsVi+OaKu24isTVTDfJd8kpUXJ0PC2Z6/DKVVNiBjym4qrvthhxhQQ+jtY9NDkZmPN1cS/XoFvYybZppP/XmeHdAMki7iWCYhQ/WiWZ7nnQeBIuDFp6bWHjmXfXgI+0EUlt/EMU1HQq2PfALj2h8abhI63uQhcyx9gxw0q2uW0lx9mZcs2pHcarX0gPdhXEWwY8nnX8B5oG33/m9Y269tvbqGwbb0OeeuDljTFUYbaM0i+CvnKFk297FcnwdYtd0x4GTzUk2uftD309JpTsIDbpjUCXJm01Yryty986lO+UrEV1KO6yeHWRrNPLc/nL78e3Pb39ilHRmKgYmaTsIWpiByodFllDMgiSoVAewSgXTYMbYis6hch7cvghqVvm9Keo0SWV+Zuh4icx2PWI81HDc4/o2gUIcAag/l/ddXTs4vGmZ6kJhewt0bq93iypNzg2SrW0M2BE/Qxehaf4cfOm0SjXWdeR0C3VXXFpMbhCYlWZ9VwiKydU1GfFzr061wIExDLq4tEDdQ6ePd1OaH7jiHgicnHTLm2rL3HW9uuiqM8dhWEfUt/GZlg84iDgFEO2jreXFRems3ZJv5KH7AFXNoAEaG0+ukMtSwpACoP/dAHS7uM8ZKVvm6s/IfGotYWJri2uwgtMsM4B8t9V7O2HTBQbdRh7y8n8YGdWhEnlIFIaEF8KwrfAP9eZ/R6WihaG65k1prf/uI1oluY7LiWaTnQ0GPSr/8Y6vE7buMEisWl+vkZMSnDeDsmsH4DHGH4J8Va/t7jJskBht0geI1rsmUquMgHkSS0EfdJugrCiDw8s5LbXaez4o41zf61l8LSAs76WFUw0+mupMX49xT0RdY+4aX+qxlr5E49pMC9F4+Stev8/HrJ9urVNMfMXkiyiLwE+C1ik5dVoBpk7TUjdNnRK38JL0Dtofg652Q/NcvJrCMCxCF2fUTntsl4e8I0oH96pXP0eZIhd+1Ofd/s8HeL1fb99/Np6uN3q42rZ6GZhrck8r3hlmj914gZhch3dyp28s3Pa+E0dwLR+SdAWO2wqq4U8S1m2zXwIDedBfzAk8zoZ5bXcLNi3I1/VZtL7hOToyEWGgb48Ob46wWouiA1DLBwTWcEPVE4rDY49V4HSSTxWngZRIFsfS4EHdDAvtVaNPoTN3GMJpj0MN5eKQxj+2hh7Fxhcyu7Z/yPgwM2ifxPMUAkZ6krfXdKcwzZNj6z1pXZGlORtdWZN1lcYq0FtggOsfKkzf1rdS1jzrkUd0dji4v7uDu3gabBhbParF2cX8ie5ny6ZW/F3AkUvaAVt7wFPvRlCWd/TykA0aUUcnaTDsIIs2iySyKrp0p6vtub57d4eXia+vEWe0nGbD/Zir60D3tTx7CPdU6qH70N5EDL0wg0aIFdiWJCcd6HN78+WmPbKjtiLXD9q7BNPN7VyBVieFsWdF634CoR+fjdCL97tSZ4B2eW18h1mCQidcjq/nCAZwaMDhAOy4B9kxf8yh7oLB5xhcnvlhQnUdLsu2X2NMzScMs8E3DEjSl/V9xLdDqwKzUgaaQSJxRGlK+sPlAE/PHG0d5nZYf10wngjv2hXO3sWxXkl3vjzs7G6VL4U3JsCm4TkQ18GNshKt6xMky2d3yIS1lghUf0ZdR5RR0sdOh8o4LLlpug4disfubuoZfBJDTHS0OvdP4YUnrmARzocL6SMHRfMuinTx3PaNEKmItTQFLDZlm0Vk+1pS7Ita0KqdGKEPQzwb1MbmfwUWywChCI3AT9NnY4RUCBRtNg1A2g6gjxPBIURQhMTMx1eWDrDW3gdKQwiBs0vfsw3I0f57Q6VCPqQAeNKCeGGuqqUnvGC+r0OgtV6+fmmBGRCZj4gYzChzkNvvyiL9ARyX4u8L+LlEUtGYPpKLixIYAhP/J/l/A2Cshj5aixLsvibQoHHlL3xALML6AJ8NgIim0V+cyQrEZPb0SZ2iBC4BZ3WaWTJJGT1VEtMSFNrKnHANxUnsFUNTtYZo598CHlXE+Z75Ls3YrLYNuuCBi4A/L/U9CjlxTuaqVVYsHPvE5jvHkpJHGmmySjsMy10cxWsgWZsFA7qSSvm1BlTUVHp5tIGDzBNFxTLH5n5bz6A7/xLYn27+68vtLzcf39ye6azw1mSC829s55sEOF2cPKbArgitG068kzCpuICvs2/eLSySsgzCtqqunBIeMtNgDgkMLeLitC12NY0s+RBSmlhz3RPH9CAjMViaRWfCGK/i8hk0dwO/3rx/8/bGXIqHn28//vr2/a09vBsAXt8OobpWUBtFSQl3fwn6E/skZK7gzMJIxWkaMPI8uFwwfldPNMIEYUgyD8MggP7I1qA8X5cTRv8LXiDXzaYqAAA='}
SHAS={'quotidien-prepare.jsonl': '370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c', 'AUDIT-DONNEES.json': '0959e340b67944f8333f426465c79c3e6b04ca9717597f1bf5e31e5f5338d643', 'CREDITS.txt': '1b2f08cabd8d7cae0f6352bebba760385b2471adf8366788c5a592cdeefabedd', 'wiktionnaire-source.jsonl': 'fced90dc06d35a6bcb10d0c6588b843a4808e47bcdf02ba876b483d6df839256', 'mandia_stage5.py': '023dded2cfec26496f667d70bb73064a8e87c8f934e16f87b67ff9454e81d2d7'}
for name,b in ASSETS.items():
    data=gzip.decompress(base64.b64decode(b));assert hashlib.sha256(data).hexdigest()==SHAS[name]
    p=OUT/name
    if p.exists():assert hashlib.sha256(p.read_bytes()).hexdigest()==SHAS[name], 'Sources modifiées : utiliser un nouveau dossier de run.'
    else:p.write_bytes(data)
sys.path.insert(0,str(PARENT/'code'));sys.path.insert(0,str(OUT))
for name in ['mandia_stage4','mandia_stage5']:sys.modules.pop(name,None)
import mandia_stage5
report=mandia_stage5.main()


## Lire le résultat
Le rapport `rapport-etape5.json` indique les mesures avant/après et si un nouveau checkpoint a été retenu. Si aucun candidat respecte les critères, le modèle de départ est conservé.


In [ ]:
print(json.dumps(report,ensure_ascii=False,indent=2))


## Essayer le candidat expérimental
Le mode reply conserve trois intentions apprises ; pour une phrase, employer translate ou une demande guidée « Dis en baoulé : ... ». Les autres réponses restent peu fiables.


In [ ]:
import torch,json,sys
from tokenizers import Tokenizer
sys.path.insert(0,str(PARENT/'code'))
from mandia_seq2seq import MandiaS2S,S2SConfig,generate
payload=torch.load(OUT/'model.pt',map_location='cpu',weights_only=True)
tok=Tokenizer.from_file(str(PARENT/'tokenizer.json'))
assert hashlib.sha256((PARENT/'tokenizer.json').read_bytes()).hexdigest()==payload['tokenizer_sha256']
model=MandiaS2S(S2SConfig(**payload['config'])).to('cuda').eval();model.load_state_dict(payload['state_dict'])
def repondre(texte,mode='reply'):
    assert mode in ['reply','translate']
    ids=[tok.token_to_id('<'+mode+'>')]+tok.encode(texte).ids+[2]
    if len(ids)>160:raise ValueError('Texte trop long pour ce prototype.')
    with torch.autocast('cuda',dtype=torch.bfloat16):answer,_=generate(model,ids,1,2,96)
    return tok.decode(answer)
for text in ['Bonjour !','Comment vas-tu ?','Au revoir']:
    print(text,'→',repondre(text))
